<a href="https://colab.research.google.com/github/Ryan56-56/project1ML/blob/main/bestsofar(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import imageio.v2 as imageio
from tqdm import tqdm

# =====================
# Config
# =====================
BATCH_SIZE   = 128
LR           = 0.001
EPOCHS       = 50
WEIGHT_DECAY = 1e-4

# IMPORTANT: dataset copied locally for speed
train_path = "/content/cifar_local/CIFAR-10-images-master/train"
test_path  = "/content/cifar_local/CIFAR-10-images-master/test"

CKPT_DIR = "/content/sample_data/CNN_model_CIFAR10"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [16]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
    T.RandomErasing(p=0.1)
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616])
])

In [19]:
train_path = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train"
test_path  = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test"

class CIFARFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.classes = sorted(os.listdir(root))

        for label_idx, folder in enumerate(self.classes):
            folder_path = os.path.join(root, folder)
            for img_name in os.listdir(folder_path):
                self.samples.append((os.path.join(folder_path, img_name), label_idx))

        print(f"Found {len(self.samples)} images in {root}.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = imageio.imread(img_path)

        if self.transform:
            img = self.transform(img)

        return img, label

train_ds = CIFARFolderDataset(train_path, transform=train_transform)
test_ds  = CIFARFolderDataset(test_path,  transform=test_transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=4, pin_memory=True, persistent_workers=True)

test_dl = DataLoader(test_ds, batch_size=256, shuffle=False,
                     num_workers=4, pin_memory=True, persistent_workers=True)

Found 50001 images in /content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train.
Found 10000 images in /content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test.


In [20]:
class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = BetterCNN().to(device)
model = model.to(memory_format=torch.channels_last)
model = torch.compile(model)

In [21]:
opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda")

In [22]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            preds = model(xb)
            loss = loss_fn(preds, yb)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_dl):.4f}")


def evaluate():
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
            preds = model(xb).argmax(1).cpu()

            preds_all.extend(preds.numpy())
            labels_all.extend(yb.numpy())

    acc = accuracy_score(labels_all, preds_all)
    prec = precision_score(labels_all, preds_all, average='weighted', zero_division=0)
    rec = recall_score(labels_all, preds_all, average='weighted', zero_division=0)
    f1 = f1_score(labels_all, preds_all, average='weighted', zero_division=0)

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    return f1

In [ ]:
resume_path = os.path.join(CKPT_DIR, "last_epoch.pth")
start_epoch = 0

if os.path.exists(resume_path):
    print("Resuming from last checkpoint...")
    checkpoint = torch.load(resume_path)
    model.load_state_dict(checkpoint["model"])
    opt.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resumed at epoch {start_epoch}")

best_f1 = 0.0

for epoch in range(start_epoch, EPOCHS):
    train_one_epoch(epoch)
    scheduler.step()

    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict()
    }, resume_path)

    print(f"Checkpoint saved: {resume_path}")

    if (epoch + 1) % 5 == 0:
        print("Evaluating...")
        f1 = evaluate()
        if f1 > best_f1:
            best_f1 = f1
            best_path = os.path.join(CKPT_DIR, "best_model.pth")
            torch.save(model.state_dict(), best_path)
            print(f"New best model saved with F1 = {f1:.4f}")

print("Final evaluation:")
final_f1 = evaluate()
print(f"Best F1 during training: {best_f1:.4f}")

Epoch 1/50:  10%|▉         | 38/391 [08:27<1:00:28, 10.28s/it, loss=1.7] 

In [ ]:
ckpt_path = "/content/sample_data/CNN_model_CIFAR10/best_model.pth"

model = BetterCNN().to(device)
model = model.to(memory_format=torch.channels_last)
model.load_state_dict(torch.load(ckpt_path))
model = torch.compile(model)

print("Loaded model:", ckpt_path)
print("Running test set evaluation...\n")

evaluate()

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
